# Document load

In [17]:
# https://python.langchain.com/docs/integrations/document_loaders/sitemap/#fix-notebook-asyncio-bug
import nest_asyncio

nest_asyncio.apply()

## www.getdbt.com

In [2]:
from langchain_community.document_loaders.sitemap import SitemapLoader

getdbt_sitemap_loader = SitemapLoader(web_path="https://www.getdbt.com/sitemap-0.xml")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
docs = getdbt_sitemap_loader.load()

Fetching pages: 100%|################################################################################################################| 1263/1263 [11:29<00:00,  1.83it/s]


## docs.getdbt.com

In [9]:
docs_getdbt_sitemap_loader = SitemapLoader(web_path="https://docs.getdbt.com/sitemap.xml")
docs_getdbt_docs = docs_getdbt_sitemap_loader.load()

Fetching pages: 100%|################################################################################################################| 1271/1271 [03:01<00:00,  7.00it/s]


# Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)

getdbt_sitemap_texts = text_splitter.split_documents(docs)
docs_getdbt_sitemap_texts = text_splitter.split_documents(docs_getdbt_docs)

In [24]:
print(len(docs))
print(len(getdbt_sitemap_texts))
print(len(docs_getdbt_docs))
print(len(docs_getdbt_sitemap_texts))

dbt_all_texts = getdbt_sitemap_texts + docs_getdbt_sitemap_texts

print(len(dbt_all_texts))


1263
8772
1271
12651
21423


# Embedding

Bedrock embedding を利用する

In [18]:
from langchain_chroma import Chroma
from langchain_aws import BedrockEmbeddings

config = {
    "region_name": "us-west-2",
    "embedding_model_id": "amazon.titan-embed-text-v2:0",
    "temperature": 0.4
}

embeddings = BedrockEmbeddings(
    model_id=config["embedding_model_id"], region_name=config["region_name"], client=None
)

LangChain で簡単に使える vector DB Chroma を利用する。collection name と sqlite3 保存パスを指定する

https://docs.trychroma.com/docs/overview/getting-started

In [ ]:
vector_store = Chroma(
    collection_name="dbt_all_docs_20250603",
    embedding_function=embeddings,
    persist_directory="./chroma_dbt_langchain_db",
)

Document 追加時に Bedrock で Embedding するが 21423 chunks で 1h 程度掛かった

In [ ]:
from itertools import batched

CHROMA_MAX_BATCH_SIZE = 5461

for batch_texts in batched(dbt_all_texts, CHROMA_MAX_BATCH_SIZE):
    vector_store.add_documents(batch_texts)

In [43]:
results = vector_store.similarity_search_with_score(
    "What is dbt fusion?",
    k=3
)
for res in results:
    print("========")
    print(res)

(Document(id='2ee839c4-a6e6-4bce-8fc3-7efa76d718a0', metadata={'changefreq': 'weekly', 'loc': 'https://docs.getdbt.com/guides/fusion', 'priority': '0.5', 'source': 'https://docs.getdbt.com/guides/fusion'}, page_content='Fusion enables you to compile and run your dbt projects faster than ever. We understand that you may want to see Fusion in action for yourself before you try it out in your development and production environments, and this quickstart guide aims to do just that!About the dbt Fusion engine\u200bFusion and the powerful features that the engine provides are available in the following:'), 0.4227229356765747)
(Document(id='f065014c-8165-4452-94d3-2172fe24409a', metadata={'changefreq': 'weekly', 'loc': 'https://docs.getdbt.com/docs/dbt-versions/product-lifecycles', 'priority': '0.5', 'source': 'https://docs.getdbt.com/docs/dbt-versions/product-lifecycles'}, page_content="dbt Core: The open-source software that’s freely available.\ndbt: The cloud-based SaaS solution, originally

Embedding した結果の persist directory を参照することで再 embedding は不要となる。

In [2]:
!tree ./chroma_dbt_langchain_db

./chroma_dbt_langchain_db
├── 4bd22bcf-6a8d-40b0-ba3f-c6de9598c42c
│   ├── data_level0.bin
│   ├── header.bin
│   ├── index_metadata.pickle
│   ├── length.bin
│   └── link_lists.bin
├── 6052dfe0-eda3-4f71-ba7e-274dd2ee30fc
└── chroma.sqlite3

3 directories, 6 files


In [ ]:
vector_store_with_persist = Chroma(
    collection_name="dbt_all_docs_20250603",
    embedding_function=embeddings,
    persist_directory="./chroma_dbt_langchain_db",
)

In [ ]:
results = vector_store_with_persist.similarity_search_with_score(
    "What is dbt fusion?",
    k=3
)
for res in results:
    print("========")
    print(res)

# Archive

MCP server で使い回すために圧縮する

In [55]:
!tar zcf chroma_dbt_langchain_db.tar.gz ./chroma_dbt_langchain_db